<a href="https://colab.research.google.com/github/Libelle210/Coffee_Finance/blob/notebooks/Download_GEE_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.6/479.6 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: earthengine-api
    Found existing installation: earthengine-api 1.7.22
    Uninstalling earthengine-api-1.7.22:
      Successfully uninstalled earthengine-api-1.7.22


In [ ]:
import ee
import time

# 🛠️ 【请在这里贴入你刚刚复制的项目名称】
MY_PROJECT = 'resolute-might-595811-a3'

# 验证并启动 GEE
try:
    # 尝试直接带项目名初始化
    ee.Initialize(project=MY_PROJECT)
except Exception as e:
    # 如果没认证过，先进行网页认证
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT)

# ----------------- 以下代码保持不变 -----------------
# 获取你当前账户下所有的任务列表
tasks = ee.batch.Task.list()

print("正在检测网页端未提交的任务...")

count = 0
for task in tasks:
    if task.state == 'UNSUBMITTED':
        task_name = task.config.get('description', '未命名任务')
        print(f"正在启动任务: {task_name}")
        task.start()
        count += 1
        time.sleep(1.5)

if count == 0:
    print("没有找到需要运行的任务。")
else:
    print(f"\n🎉 大功告成！已成功将 {count} 个任务提交至云端排队！")

正在检测网页端未提交的任务...
没有找到需要运行的任务。


In [ ]:
import ee
import time

# ══════════════════════════════════════════════════════════════
# §0  配置与认证
# ══════════════════════════════════════════════════════════════

# 填入你 GEE 网页端右上角显示的 Cloud Project 名称
MY_PROJECT = 'resolute-might-595811-a3'

try:
    ee.Initialize(project=MY_PROJECT)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT)

DRIVE_FOLDER = "GEE_Coffee"      # Google Drive目标文件夹
START_YEAR   = 2022              # 起始年
END_YEAR     = 2024              # 结束年

# 季度定义
QUARTERS = [
  {"name": "Q1", "start_month": 1,  "end_month": 3},
  {"name": "Q2", "start_month": 4,  "end_month": 6},
  {"name": "Q3", "start_month": 7,  "end_month": 9},
  {"name": "Q4", "start_month": 10, "end_month": 12},
]

# ══════════════════════════════════════════════════════════════
# §1  产区列表 (与你的原 JS 脚本完全一致)
# ══════════════════════════════════════════════════════════════
sites = [
  # ── 越南产区 ──────────────────────────────────────────────
  {"id": "VN1", "csvName": "多乐邦美蜀_EaTu",  "lon": 108.1002, "lat": 12.7105, "buffer": 15000},
  {"id": "VN2", "csvName": "林同大叻_CauDat",  "lon": 108.5372, "lat": 11.8516, "buffer": 15000},
  {"id": "VN3", "csvName": "嘉莱_ChưPrông",   "lon": 107.9152, "lat": 13.7548, "buffer": 10000},
  {"id": "VN4", "csvName": "得农_DakMil",     "lon": 107.6523, "lat": 12.4512, "buffer": 15000},
  {"id": "VN5", "csvName": "林同保禄_BaoLoc", "lon": 107.8213, "lat": 11.5124, "buffer": 15000},

  # ── 中国云南产区 ───────────────────────────────────────────
  {"id": "YN1", "csvName": "大开河_林润庄园",  "lon": 100.9833, "lat": 22.3833, "buffer": 10000},
  {"id": "YN2", "csvName": "新寨咖啡庄园",     "lon": 98.5667,  "lat": 25.0500, "buffer": 10000},
  {"id": "YN3", "csvName": "怒江粒述庄园",     "lon": 98.2833,  "lat": 25.8333, "buffer": 10000},
  {"id": "YN4", "csvName": "爱伲咖啡庄园",     "lon": 100.8833, "lat": 22.5167, "buffer": 10000},
  {"id": "YN5", "csvName": "宾川朱苦拉咖啡林", "lon": 100.4500, "lat": 25.7167, "buffer": 10000},

  # ── 老挝产区 ──────────────────────────────────────────────
  {"id": "LA1", "csvName": "巴松_PaksongA",   "lon": 106.2312, "lat": 15.1823, "buffer": 12000},
  {"id": "LA2", "csvName": "DaoHeuang",       "lon": 106.1123, "lat": 15.1154, "buffer": 12000},
  {"id": "LA3", "csvName": "Thateng",         "lon": 106.3812, "lat": 15.4412, "buffer": 12000},
  {"id": "LA4", "csvName": "DakCheung",       "lon": 107.2513, "lat": 15.3512, "buffer": 12000},
  {"id": "LA5", "csvName": "Champasak",       "lon": 106.4912, "lat": 15.0712, "buffer": 12000},

  # ── 日本冲绳 ──────────────────────────────────────────────
  {"id": "JP1", "csvName": "冲绳又吉",         "lon": 128.1435, "lat": 26.6098, "buffer": 8000},
]

# ══════════════════════════════════════════════════════════════
# §2  Sentinel-2 处理函数 (翻译为 Python 语法)
# ══════════════════════════════════════════════════════════════

def maskS2Clouds(image):
    scl = image.select("SCL")
    # Python API 中逻辑“或”使用 .Or()
    clearMask = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7)).Or(scl.eq(11))
    return image.updateMask(clearMask)

def selectBands(image):
    return image.select(
        ["B2", "B3", "B4", "B8", "B11", "B5", "SCL"],
        ["blue", "green", "red", "nir", "swir1", "rededge", "SCL"]
    )

def getQuarterlyComposite(roi, year, quarter):
    start = ee.Date.fromYMD(year, quarter["start_month"], 1)
    end = ee.Date.fromYMD(year, quarter["end_month"], 28).advance(4, "day")

    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(roi)
           .filterDate(start, end)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
           .map(maskS2Clouds)
           .map(selectBands))

    # 用 Python 预先检查该季度是否有影像，若无则跳过，避免生成空白任务
    if col.size().getInfo() > 0:
        return col.median().clip(roi)
    else:
        return None

# ══════════════════════════════════════════════════════════════
# §3  直接由 Python 循环并自动向云端提交 Task
# ══════════════════════════════════════════════════════════════

years = [START_YEAR, START_YEAR + 1, END_YEAR]  # [2022, 2023, 2024]
task_count = 0

print("正在直接通过 API 批量生成并提交 Sentinel-2 任务...")

for site in sites:
    roi = ee.Geometry.Point([site["lon"], site["lat"]]).buffer(site["buffer"])

    for year in years:
        for quarter in QUARTERS:

            composite = getQuarterlyComposite(roi, year, quarter)

            if composite is not None:
                taskName = f"S2_{site['id']}_{year}_{quarter['name']}"

                # 使用 Python API 的 ee.batch.Export 提交
                task = ee.batch.Export.image.toDrive(
                    image = ee.Image(composite).toFloat(),
                    description = taskName,
                    folder = DRIVE_FOLDER,
                    fileNamePrefix = taskName,
                    region = roi,
                    scale = 10,
                    crs = "EPSG:4326",
                    maxPixels = int(1e10)
                )

                # 核心步骤：直接启动任务！不再依赖网页端的 RUN 按钮
                task.start()
                task_count += 1
                print(f"✅ 已成功提交任务: {taskName}")

                # 稍微设置间隔，防止向 Google 服务器发送请求过快
                time.sleep(1.0)
            else:
                print(f" ⏭️ 产区 {site['id']} 在 {year} {quarter['name']} 无有效无云影像，已跳过。")

print(f"\n🎉 运行完毕！已成功通过 Python 为你向云端提交了 {task_count} 个 Sentinel-2 导出任务！")

正在直接通过 API 批量生成并提交 Sentinel-2 任务...
✅ 已成功提交任务: S2_VN1_2022_Q1
✅ 已成功提交任务: S2_VN1_2022_Q2
✅ 已成功提交任务: S2_VN1_2022_Q3
✅ 已成功提交任务: S2_VN1_2022_Q4
✅ 已成功提交任务: S2_VN1_2023_Q1
✅ 已成功提交任务: S2_VN1_2023_Q2
✅ 已成功提交任务: S2_VN1_2023_Q3
✅ 已成功提交任务: S2_VN1_2023_Q4
✅ 已成功提交任务: S2_VN1_2024_Q1
✅ 已成功提交任务: S2_VN1_2024_Q2
✅ 已成功提交任务: S2_VN1_2024_Q3
✅ 已成功提交任务: S2_VN1_2024_Q4
✅ 已成功提交任务: S2_VN2_2022_Q1
✅ 已成功提交任务: S2_VN2_2022_Q2
✅ 已成功提交任务: S2_VN2_2022_Q3
✅ 已成功提交任务: S2_VN2_2022_Q4
✅ 已成功提交任务: S2_VN2_2023_Q1
✅ 已成功提交任务: S2_VN2_2023_Q2
✅ 已成功提交任务: S2_VN2_2023_Q3
✅ 已成功提交任务: S2_VN2_2023_Q4
✅ 已成功提交任务: S2_VN2_2024_Q1
✅ 已成功提交任务: S2_VN2_2024_Q2
✅ 已成功提交任务: S2_VN2_2024_Q3
✅ 已成功提交任务: S2_VN2_2024_Q4
✅ 已成功提交任务: S2_VN3_2022_Q1
✅ 已成功提交任务: S2_VN3_2022_Q2
✅ 已成功提交任务: S2_VN3_2022_Q3
✅ 已成功提交任务: S2_VN3_2022_Q4
✅ 已成功提交任务: S2_VN3_2023_Q1
✅ 已成功提交任务: S2_VN3_2023_Q2
✅ 已成功提交任务: S2_VN3_2023_Q3
✅ 已成功提交任务: S2_VN3_2023_Q4
✅ 已成功提交任务: S2_VN3_2024_Q1
✅ 已成功提交任务: S2_VN3_2024_Q2
✅ 已成功提交任务: S2_VN3_2024_Q3
✅ 已成功提交任务: S2_VN3_2024_Q4
✅ 已成功提交任务: S2_VN4_2022_Q1
✅ 